Run: docker compose -f graph-docker-compose.yml up -d

In [ ]:
from neo4j import GraphDatabase
# from tabulate import tabulate
import pandas as pd

In [ ]:
# Połączenie z Neo4j
URI = "bolt://neo4j_nosql_lab:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"  # upewnij się, że to pasuje do graph-docker-compose.yml

In [ ]:
def run_cypher(driver, query: str):
    with driver.session() as session:
        result = session.run(query)
        records = list(result)

        if not records:
            print("Brak wyników.")
            return

        df = pd.DataFrame([r.data() for r in records])
        return df


def qexec(query: str):
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

    try:
        df = run_cypher(driver, query)
    finally:
        driver.close()

    return df

In [ ]:
def reset(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def load_sample_data(tx):
    tx.run("""
        // USERS
        CREATE (u1:User {name: 'U1'}),
        (u2:User {name: 'U2'}),
        (u3:User {name: 'U3'}),
        (u4:User {name: 'U4'}),
        (u5:User {name: 'U5'}),

        // DEVICES
        (d1:Device {id: 'D1'}),
        (d2:Device {id: 'D2'}),
        (d3:Device {id: 'D3'}),

        // NORMAL
        (u1)-[:USES]->(d1),
        (u2)-[:USES]->(d2),

        // FRAUD CLUSTER
        (u3)-[:USES]->(d3),
        (u4)-[:USES]->(d3),
        (u5)-[:USES]->(d3),

        // TRANSFERS
        (u3)-[:TRANSFER {amount: 1000}]->(u1),
        (u4)-[:TRANSFER {amount: 2000}]->(u2)
    """)

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    session.execute_write(reset)
    session.execute_write(load_sample_data)

print("Dane zostały załadowane!")

driver.close()


### Ćwiczenie 1: Shared Device Detection
Fraud: wiele kont korzysta z jednego urządzenia.
Znajdź urządzenia używane przez >2 userów.

In [ ]:
q_cypher = """
// Cypher
// grupujemy po device
MATCH (d:Device)<-[:USES]-(u:User)
WITH d, COUNT(u) AS users

// filtr fraud
WHERE users > 2

RETURN d.id, users;
"""

q_gds_1 = """
// GDS
// projekcja grafu
CALL gds.graph.project(
  'g1',
  ['User', 'Device'],
  { USES: { orientation: 'UNDIRECTED' } }
)
"""

q_gds_2 = """
// community detection
CALL gds.louvain.stream('g1')
YIELD nodeId, communityId

RETURN
  gds.util.asNode(nodeId).name,
  communityId;
"""

In [ ]:
qexec(q_cypher)

In [ ]:
qexec(q_gds_1)
qexec(q_gds_2)

### Ćwiczenie 2: Multi-hop powiązania
Fraud często ukrywa się w powiązaniach pośrednich.
Czy U3 jest powiązany z U2?

*..4 = maksymalnie 4 hop-y

In [ ]:
q_cypher = """
// Cypher
// szukamy dowolnej ścieżki do długości 4
MATCH path =
  (u3:User {name: 'U3'})-[*..4]-(u2:User {name: 'U2'})

RETURN path;
"""

In [ ]:
qexec(q_cypher)

### Ćwiczenie 3: Shortest Path (Cypher vs GDS)
Najkrótsza ścieżka = najsilniejsze powiązanie.
Znajdź shortest path między U3 i U2.

In [ ]:
q_cypher = """
// Cypher
// shortestPath znajduje tylko 1 ścieżkę
MATCH path = shortestPath(
  (u3:User {name: 'U3'})-[*..5]-(u2:User {name: 'U2'})
)

RETURN path;
"""

q_gds_1 = """
// GDS
// projekcja
CALL gds.graph.drop('sp');
"""

q_gds_2 = """
CALL gds.graph.project(
  'sp',
  ['User', 'Device'],
  {
    USES: { orientation: 'UNDIRECTED' },
    TRANSFER: { orientation: 'UNDIRECTED' }
  }
);
"""

q_gds_3 = """
MATCH (s:User {name: 'U3'})
MATCH (t:User {name: 'U2'})

CALL gds.shortestPath.dijkstra.stream('sp', {
  sourceNode: id(s),
  targetNode: id(t)
})
YIELD nodeIds

RETURN
  [n IN nodeIds | gds.util.asNode(n).name] AS path;
"""

In [ ]:
qexec(q_cypher)

In [ ]:
qexec(q_gds_1)
qexec(q_gds_2)
qexec(q_gds_3)

### Ćwiczenie 4: Fraud Scoring
System musi ocenić ryzyko.
Policz: fraud_score = 1 / path_length + shared_device_bonus

In [ ]:
q_cypher = """
MATCH path = shortestPath(
  (u3:User {name: 'U3'})-[*..5]-(u2:User {name: 'U2'})
)

WITH path, length(path) AS len

// sprawdzamy wspólny device
OPTIONAL MATCH (u3:User {name: 'U3'})-[:USES]->(d)<-[:USES]-(u2:User {name: 'U2'})

WITH path, len,
  CASE WHEN d IS NOT NULL THEN 1 ELSE 0 END AS bonus

RETURN
  path,
  1.0 / len + bonus AS fraud_score;
"""

In [ ]:
qexec(q_cypher)